<a href="https://colab.research.google.com/github/xc308/Dataset_preparation_Fine_Tuning/blob/main/LoRA_Pars_Efficient_FT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**:

Apply LoRA (Low-Rank Adaptation) for parameter-efficient fine-tuning.

In [1]:
# Install the custom package for this course.
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"

import os # For setting Keras configuration variables.

# The following two lines provide configuration for Keras.
os.environ["KERAS_BACKEND"] = "jax"

import keras # For building a model.
from keras import layers # As base class for a custom LoRA layer.
keras.utils.set_random_seed(812) # For Keras layers.

import jax # For working with vectors and matrices.
import jax.numpy as jnp # For working with vectors and matrices.

# For checking your solutions.
from ai_foundations.feedback.course_5 import lora as feedback

  Cloning https://github.com/google-deepmind/ai-foundations.git (to revision main) to /tmp/pip-req-build-18ncufh8
  Running command git clone --filter=blob:none --quiet https://github.com/google-deepmind/ai-foundations.git /tmp/pip-req-build-18ncufh8
  Resolved https://github.com/google-deepmind/ai-foundations.git to commit 524d6114bbce631dafc00ba3496607a0bc60c804
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
class SimplifiedDense(layers.Layer):
    """A simplified implementation of a dense layer.

    Args:
      output_dim: The number of neurons in the layer.
      input_dim: The number of inputs to each neuron.
    """
    def __init__(self, output_dim: int = 32, input_dim: int = 32):
        """ Initializes a Dense layer."""
        super().__init__() # Call the constructor of the base class.

        self.w = self.add_weight( # w is a weight matrix.
            shape=(input_dim, output_dim), # Shape of w.
            initializer="random_normal", # Intialize the weights randomly.
            trainable=True # w shall be updated during training.
        )
        # Add b as bias weights, which is a vector.
        self.b = self.add_weight(
            shape=(output_dim,),
            initializer="zeros",
            trainable=True
        )

    def call(self, inputs: jax.Array):
        """Multiplies inputs with weight matrix and adds bias.

        Args:
          inputs: The input to the layer.

        Returns:
          The output of the layer.
        """

        # Compute y = XW + b
        return jnp.matmul(inputs, self.w) + self.b

In [2]:
class SimplifiedDense(layers.Layer):
    """A simplified dense layer

    Arg:
      output_dim: The number of neurons in the layer.
      input_dim: The number of inputs to each neuron.
    """

    def __init__(self, output_dim: int = 32, input_dim: int = 32):
        """Initializes a Dense layer."""

        super().__init__()

        self.w = self.add_weight(
            shape = (input_dim, output_dim),
            initializer = "random_normal",
            trainable = True
        )

        self.b = self.add_weight(
            shape = (output_dim, ),
            initializer = "zeros",
            trainable = True
        )

    def call(self, inputs: jax.Array):
        """Multiplies inputs with weight matrix and adds bias.

        Args:
            inputs: The input to the layer.

        Returns:
          The output of the layer.
        """

        return jnp.matmul(inputs, self.w) + self.b


In [5]:
class LoraDense(layers.Layer):
  """An implementation of a dense layer with LoRA.

  Args:
    pretrained_layer: The original pre-trained dense layer.
    rank: The rank of the LoRA matrices.

  Attributes:
    num_units: The number of units in the dense layer.
    rank: The rank of the LoRA matrices.
    A: The LoRA matrix A.
    B: The LoRA matrix B.
    W0: The pre-trained weights.
    bias: The pre-trained bias parameters.
  """
  def __init__(self , pretrained_layer: layers.Dense, rank: int = 8, **kwargs):
    """Initializes a LoRA layer.

    Args:
      pretrained_layer: The original pre-trained dense layer.
      rank: The rank of the LoRA matrices.
    """

    super().__init__(**kwargs)  # Call the constructor of the base class.

    # Shape of the weight matrix W of the pre-trained layer.
    pretrained_layer_shape = pretrained_layer.weights[0].shape
    # Shape of the bias in the pre-trained layer.
    bias_shape = pretrained_layer.weights[1].shape
    # Number of units = number of columns of W of the pre-trained layer.
    self.num_units = pretrained_layer_shape[1]
    # Rank of the Lora matrices.
    self.rank = rank

    # LoRA matrix A, shape rank x num_units.
    # Note that in the original LoRA paper, this matrix is initialized
    # with random numbers. This implementation also uses this convention.
    self.A = self.add_weight(
        shape=(rank, self.num_units),
        initializer=keras.initializers.RandomNormal(stddev=0.01),
        trainable=True, # These parameters should be updated during fine-tuning.
    )
    # LoRA matrix B, shape number of inputs x r.
    # Note that in the original LoRA paper, this matrix is initialized
    # with all values being set to 0. This implementation also uses this
    # convention.
    self.B = self.add_weight(
        shape=(pretrained_layer_shape[0], rank),
        initializer=keras.initializers.Zeros(),
        trainable=True, # These parameters should be updated during fine-tuning.
    )
    # Pre-trained weights W0, copied from pre-trained layer.
    self.W0 = self.add_weight(
        shape=(pretrained_layer_shape),
        initializer=keras.initializers.Constant(pretrained_layer.weights[0]),
        trainable=False, # These parameters are frozen during fine-tuning.
    )
    # Bias weights bias, copied from pre-trained layer.
    self.bias = self.add_weight(
        shape=(bias_shape),
        initializer=keras.initializers.Constant(pretrained_layer.weights[1]),
        trainable=True, # These parameters should be updated during fine-tuning.
    )

  def call(self, x: jax.Array):
    """
      Forward pass through the layer.

      Args:
        x: The input to the layer.

      Returns:
        The output of the layer.
    """

    # Compute ΔW = BA.
    delta_W = jnp.matmul(self.B, self.A)

    # Compute W = W0 + ΔW.
    W = self.W0 + delta_W

    # Compute layer output y = xW + b.
    y = jnp.matmul(x, W) + self.bias

    return y

**Test LoRA layer**

- To test the implemenation of your LoRA layer
    - implement a small one-layer MLP and
    - then replace its dense layer with a LoRA layer.

In [6]:
# Number of units for inputs and outputs.
# In a transformer, this is typically the embedding space, size of the
# attention mechanism, and the output of a transformer block.
num_units = 512
rank = 8

# Define a model with a single dense layer in pretrained_model.
x = layers.Input(shape=(num_units,))
y = layers.Dense(units=num_units, name="dense_layer_1")(x) # a fully connected layer computes y = xW + b
pretrained_model = keras.Model(inputs=x, outputs=y)

# Take the dense layer to convert to a LoRA layer.
pretrained_layer = pretrained_model.get_layer("dense_layer_1")

# Define a model with a single LoRA layer.
x = layers.Input(shape=(num_units,))
y = LoraDense(
    pretrained_layer=pretrained_layer,
    rank=rank,
    name="lora_dense_layer_1"
)(x)
lora_model = keras.Model(inputs=x, outputs=y)

# Pass an example input through both the original model and the LoRA model.
input = jnp.ones((1, num_units))
output_original = pretrained_model(input)
output_lora = lora_model(input)

# Print the shapes of the output from both models.
print(f"Shape of original model output: {output_original.shape}")
print(f"Shape of LoRA model output: {output_lora.shape}")

Shape of original model output: (1, 512)
Shape of LoRA model output: (1, 512)


In [8]:
pretrained_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_layer_1 (Dense)           │ (None, 512)            │       262,656 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 262,656 (1.00 MB)

 Trainable params: 262,656 (1.00 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
lora_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lora_dense_layer_1 (LoraDense)  │ (None, 512)            │       270,848 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 270,848 (1.03 MB)

 Trainable params: 8,704 (34.00 KB)

 Non-trainable params: 262,144 (1.00 MB)

**Compute the proportion of trainable parameters**

-

In [14]:
print((4096+512+ 4096) / 270848)

0.03213610586011342


In [16]:
answer = 0.0331

feedback.check_loralab_answer(answer, num_units=512, rank=8)

✅ Nice! Your answer looks correct.
If you consider bias weights, the answer is 0.03313999995589256.
If you don't consider bias weights, the answer is 0.03125.
